# QueryPic Stage 2: clean joint INT8 CLIP

Converts the MIT-licensed LAION ViT-B/32 checkpoint into the joint
INT8 `.tflite` the app runs on-device. **Start from a FACTORY-RESET
runtime** (Runtime menu) so no stale packages linger, then Run all.
**CPU runtime is fine.** ~30-60 min wall time.

**How to run:** factory reset, then Runtime > Run all. Download
`/content/clip_joint_b32_aeq.tflite` from the Files pane when the
gate prints PASS.


In [ ]:
%pip install -q --upgrade torch open_clip_torch onnx onnxruntime onnxscript onnx2tf tensorflow-cpu huggingface_hub Pillow ai-edge-torch torchao "numpy==2.2.6"

In [ ]:
# Smoke test: versions must import TOGETHER. Red here = stop and
# report; running on is how earlier sessions burned hours.
!python -c "import torch, open_clip; print(torch.__version__, 'ok')"

In [ ]:
%%writefile export_joint_onnx.py
"""Export the MIT-licensed LAION ViT-B/32 checkpoint as a JOINT ONNX graph.

Joint layout (int64-free: TFLite rejects int64 graph inputs and int64
ArgMax outputs — "Unknown input type: 4"):
  inputs : image [1,3,224,224] float32 (CLIP-preprocessed NCHW, exactly
           what ClipVision.preprocess emits) + text [1,77] INT32 token ids
           + eot [1] INT32 (EOT position = first 49407 at index>=1; the
           Dart tokenizer computes it — no ArgMax anywhere in the graph)
  outputs: image_emb [1,512] + text_emb [1,512] float32 (RAW, unnormalized —
           the Dart side L2-normalizes + INT8-quantizes as before) +
           scale [] float32 (logit scale, ignored by the loader).

The text tail is reimplemented (embedding + transformer + ln + one-hot
pool at the EOT index) instead of calling encode_text, whose internal
argmax would reintroduce int64. One-hot + MatMul pooling uses only
universally supported ops.

Stage 1 of the clean pipeline (runs on any torch host, no TF needed):
  Stage 2: convert_litert_dr.py (direct torch->TFLite, reference DR).

Usage:
  py tools/export_joint_onnx.py [--source BIN] [--out joint.onnx]
Exit 0 + cosine agreement report on success.
"""
import argparse
import os
import sys

import torch

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DEFAULT_SRC = os.path.join(ROOT, "querypic", "sandbox", "models",
                           "open_clip_pytorch_model.bin")
DEFAULT_OUT = os.path.join(ROOT, "querypic", "sandbox", "models",
                           "clip_joint_b32.onnx")

EOT_ID = 49407


class JointCLIP32(torch.nn.Module):
    """Both towers, one graph, raw (unnormalized) 512-d outputs.

    Int32 in, no ArgMax: tokens arrive int32 (cast to int64 only for the
    embedding Gather) and pooling masked-sums the caller-supplied int32
    EOT. Torch parity vs encode_* proven to cos=1.0 (all probes).
    """

    def __init__(self, clip):
        super().__init__()
        self.token_embedding = clip.token_embedding
        self.positional_embedding = clip.positional_embedding
        self.transformer = clip.transformer
        self.ln_final = clip.ln_final
        self.text_projection = clip.text_projection
        self.attn_mask = clip.attn_mask
        self.visual = clip.visual
        self.logit_scale = clip.logit_scale

    def forward(self, image, tokens32, eot32):
        tokens = tokens32.to(torch.int64)
        img = self.visual(image)
        cast = self.transformer.get_cast_dtype()
        x = self.token_embedding(tokens).to(cast)
        x = x + self.positional_embedding.to(cast)
        x = self.transformer(x, attn_mask=self.attn_mask)
        x = self.ln_final(x)
        # Masked-sum pool at the EOT index: Equal + Mul + ReduceSum only.
        # (Earlier revisions used argmax-gather (int64 rejected by TFLite)
        # then one-hot+bmm (strict torch.export shape inference rejects
        # it). Masked sum is mathematically identical and traces cleanly
        # under strict export, dynamo export, ONNX, and litert_torch.)
        pos = torch.arange(x.shape[1], device=x.device).unsqueeze(0)
        mask = (pos == eot32).to(x.dtype).unsqueeze(-1)
        pooled = (x * mask).sum(dim=1)
        if isinstance(self.text_projection, torch.nn.Linear):
            txt = self.text_projection(pooled)
        else:
            txt = pooled @ self.text_projection
        scale = self.logit_scale.exp().reshape(())
        return img, txt, scale


def eot_of(ids) -> int:
    row = list(ids[0] if hasattr(ids, "__getitem__") else ids)
    return row.index(EOT_ID, 1)


def cosine(a: torch.Tensor, b: torch.Tensor) -> float:
    a = a.flatten().float()
    b = b.flatten().float()
    return float((a @ b) / (a.norm() * b.norm() + 1e-12))


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--source", default=DEFAULT_SRC)
    ap.add_argument("--out", default=DEFAULT_OUT)
    ap.add_argument("--no-validate", action="store_true")
    args = ap.parse_args()

    import open_clip

    print(f"  loading {os.path.basename(args.source)} ...")
    model, _, _ = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained=args.source)
    model.eval()
    joint = JointCLIP32(model).eval()

    img = torch.zeros(1, 3, 224, 224)
    tok = open_clip.get_tokenizer("ViT-B-32")(["sunset over the ocean"])
    eot = torch.tensor([eot_of(tok.tolist())], dtype=torch.int32)
    tok32 = tok.to(torch.int32)
    assert list(img.shape) == [1, 3, 224, 224]
    assert list(tok32.shape) == [1, 77], f"text layout {tok32.shape}"

    print("  exporting joint ONNX (dynamo exporter: handles SDPA) ...")
    with torch.no_grad():
        torch.onnx.export(
            joint, (img, tok32, eot), args.out,
            input_names=["image", "text", "eot"],
            output_names=["image_emb", "text_emb", "scale"],
            dynamic_axes=None,
        )
    size = os.path.getsize(args.out)
    print(f"  wrote {os.path.basename(args.out)} ({size/1e6:.1f}MB)")

    if args.no_validate:
        return 0

    # Validate: ONNX Runtime must reproduce torch embeddings ~exactly.
    try:
        import onnxruntime as ort
    except ImportError:
        print("  SKIP validation (pip install onnxruntime to enable)")
        return 0
    import numpy as np

    torch.manual_seed(0)
    probe_img = torch.randn(1, 3, 224, 224)
    probe_ids = open_clip.get_tokenizer("ViT-B-32")(
        ["receipt from home depot"])
    probe_tok = probe_ids.to(torch.int32)
    probe_eot = torch.tensor([eot_of(probe_ids.tolist())], dtype=torch.int32)
    with torch.no_grad():
        t_img, t_txt, _ = joint(probe_img, probe_tok, probe_eot)
    sess = ort.InferenceSession(args.out,
                                providers=["CPUExecutionProvider"])
    o_img, o_txt, _ = sess.run(
        None,
        {"image": probe_img.numpy().astype(np.float32),
         "text": probe_tok.numpy().astype(np.int32),
         "eot": probe_eot.numpy().astype(np.int32)},
    )
    ci = cosine(t_img, torch.from_numpy(o_img))
    ct = cosine(t_txt, torch.from_numpy(o_txt))
    print(f"  agreement: image cos={ci:.6f} text cos={ct:.6f}")
    if min(ci, ct) < 0.999:
        print("  FAIL: export diverged from torch — do not ship")
        return 1
    print("PASS: joint ONNX matches torch; ready for Stage 2 (TFLite)")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile convert_onnx_to_tflite.py
"""Stage 2 (RUN ON LINUX/COLAB — needs TF + onnx2tf, no Windows wheels).

Converts Stage 1's joint ONNX (tools/export_joint_onnx.py) into the joint
INT8 TFLite the Flutter loader runs on-device:
  inputs : image [1,3,224,224] float32 + text [1,77] int64
  outputs: image_emb [1,512] + text_emb [1,512] + scale [] (all float32;
           the Dart side normalizes + INT8-quantizes as before)

Why INT8 dynamic-range (weights-only):
  ~4x smaller (~150MB vs 605MB fp32), 2-3x faster CPU via quantized XNNPACK
  kernels, and — critically for itel S666LN, where the fp32 MobileCLIP
  graph segfaults mid-inference — an entirely different kernel path.
  Retrieval accuracy cost is typically <2% Recall@1; tools/validate_clip.py
  proves ranking agreement before anything ships.

Colab recipe (CPU runtime is fine):
  !pip install -q onnx2tf onnxruntime tensorflow-cpu
  !python tools/convert_onnx_to_tflite.py --onnx <path> --out <path>
  then tools/validate_clip.py --tflite <path> (needs onnxruntime +
  tflite_runtime or ai-edge-litert interpreter)

Usage:
  python tools/convert_onnx_to_tflite.py [--onnx IN] [--out OUT]
Exit 0 + size report on success.
"""
import argparse
import os
import sys

DEFAULT_OUT = "clip_joint_b32_int8.tflite"


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--onnx", default=None)
    ap.add_argument("--out", default=None)
    ap.add_argument("--quantize", action="store_true", default=True)
    ap.add_argument("--no-quantize", dest="quantize", action="store_false")
    args = ap.parse_args()

    try:
        import onnx2tf
    except ImportError:
        print("FAIL: pip install onnx2tf (Linux only — no Windows wheels)")
        return 1

    onnx_path = args.onnx
    if onnx_path is None:
        root = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
        onnx_path = os.path.join(root, "querypic", "sandbox", "models",
                                 "clip_joint_b32.onnx")
    out_dir = os.path.dirname(args.out) if args.out else None
    if out_dir is None:
        out_dir = os.path.dirname(onnx_path) or "."
    want_name = os.path.basename(args.out) if args.out else DEFAULT_OUT
    if not args.quantize:
        want_name = want_name.replace("_int8", "_float32")
    before = set(os.listdir(out_dir))
    print(f"  converting {os.path.basename(onnx_path)} ...")
    # Dynamic-range (weights-only) int8: activations stay float, so no
    # calibration dataset is needed and the loader's float I/O is kept.
    # NOTE: onnx2tf only emits the dynamic-range file when
    # output_dynamic_range_quantized_tflite=True — quant_type alone just
    # changes the (always-written) float16 variant. Missing this flag is
    # why the first run produced float32+float16 only (2026-10-09).
    onnx2tf.convert(
        input_onnx_file_path=onnx_path,
        output_folder_path=out_dir,
        output_integer_quantized_tflite=False,
        output_dynamic_range_quantized_tflite=args.quantize,
        quant_type="per-channel" if args.quantize else "float32",
    )
    after = sorted(set(os.listdir(out_dir)) - before)
    fresh = [f for f in after if f.endswith(".tflite")]
    print(f"  emitted: {fresh or after}")
    # Copy the chosen artifact to the canonical name so the next stage
    # (validate_clip.py --tflite <it>) never hits a missing file.
    src_name = None
    for f in fresh:
        if args.quantize and "dynamic" in f and "range" in f:
            src_name = f
            break
    if src_name is None:
        for f in fresh:
            if "float32" in f:
                src_name = f
                break
    if src_name is None and fresh:
        src_name = fresh[0]
    if src_name is None:
        print("  FAIL: no .tflite emitted — see onnx2tf log above")
        return 1
    import shutil
    dest = os.path.join(out_dir, want_name)
    shutil.copy(os.path.join(out_dir, src_name), dest)
    size = os.path.getsize(dest)
    print(f"  canonical: {want_name} ({size/1e6:.1f}MB, from {src_name})")
    print("  NEXT: tools/validate_clip.py --tflite "
          f"{dest}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile quantize_aeq.py
"""Reference dynamic-range via ai_edge_quantizer on a FLOAT32 TFLite.

Why this file exists: onnx2tf's and litert_torch's dynamic-range passes
quantize the token-embedding table ASYMMETRIC (nonzero zero-point), and
every TFLite embedding_lookup kernel — Colab, Android, everywhere —
rejects it (`zero_point == 0 was not true`, node 758). Google's reference
quantizer (TFLiteConverter heritage) emits SYMMETRIC weight tables, which
the kernel accepts. Same float I/O, same ~4x size win.

Usage (Linux/Colab):
  !python quantize_aeq.py --in /content/clip_joint_b32_float32.tflite \
      --out /content/clip_joint_b32_aeq.tflite
Exit 0 + byte count on success.
"""
import argparse
import os
import sys


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--in", dest="src", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args()

    from ai_edge_quantizer import quantizer, recipe

    q = quantizer.Quantizer(args.src)
    q.update_quantization_recipe(recipe.dynamic_wi8_afp32())
    q.quantize()
    q.save(args.out)
    size = os.path.getsize(args.out)
    print(f"  wrote {os.path.basename(args.out)} ({size/1e6:.1f}MB)")
    print("  NEXT: tools/validate_clip.py --tflite "
          f"{args.out}")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile validate_clip.py
"""Ranking-agreement gate for the converted joint TFLite (Stage 2 exit).

Compares the INT8 TFLite against the float PyTorch source on a probe set
of real photos x captions:
  * per-tower cosine drift (tflite vs torch embeddings)
  * image->caption top-1 agreement over the cross-similarity matrix

PASS: top-1 agreement >= 7/8 AND mean abs drift < 0.05.
Dynamic-range int8 on retrieval towers is usually near-exact; anything
worse means the conversion damaged the space — do not ship.

No torchvision anywhere (hand-rolled PIL preprocess): torchvision is
version-coupled to torch and breaks across environments.

Usage:
  python tools/validate_clip.py --tflite clip_joint_b32_int8.tflite
      [--source open_clip_pytorch_model.bin]
Exit 0 on PASS.
"""
import argparse
import math
import os
import sys
import urllib.request

import numpy as np

CAPTIONS = [
    "a photo of a cat",
    "sunset over the ocean",
    "receipt from a grocery store",
    "a dog running on grass",
    "city street at night",
    "birthday cake with candles",
    "mountain lake reflection",
    "busy market stall with fruit",
]

# Real photos (deterministic seeds) so text<->image similarities are real.
IMAGE_URLS = [
    f"https://picsum.photos/seed/querypic{i}/400/400" for i in range(8)
]

CLIP_MEAN = np.array([0.48145466, 0.4578275, 0.40821073], dtype=np.float32)
CLIP_STD = np.array([0.26862954, 0.26130258, 0.27577711], dtype=np.float32)


def clip_preprocess(pil_img):
    """Center-crop, resize 224, normalize — mirrors ClipVision.preprocess."""
    from PIL import Image

    w, h = pil_img.size
    side = min(w, h)
    img = pil_img.crop(((w - side) // 2, (h - side) // 2,
                        (w + side) // 2, (h + side) // 2))
    img = img.resize((224, 224), Image.BICUBIC).convert("RGB")
    arr = np.asarray(img).astype(np.float32) / 255.0
    arr = (arr - CLIP_MEAN) / CLIP_STD
    return arr.transpose(2, 0, 1)[None, :, :, :]  # [1,3,224,224]


def fetch_images(cache_dir):
    os.makedirs(cache_dir, exist_ok=True)
    paths = []
    for i, url in enumerate(IMAGE_URLS):
        dest = os.path.join(cache_dir, f"probe_{i}.jpg")
        if not os.path.exists(dest):
            req = urllib.request.Request(url,
                                         headers={"User-Agent": "QueryPic/1.0"})
            with urllib.request.urlopen(req, timeout=60) as r, \
                    open(dest, "wb") as f:
                f.write(r.read())
        paths.append(dest)
    return paths


def norm(v):
    v = v.astype(np.float64)
    return v / (np.linalg.norm(v) + 1e-12)


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--tflite", required=True)
    ap.add_argument("--source", default=None)
    ap.add_argument("--cache", default="/tmp/clip_probes")
    args = ap.parse_args()

    import torch
    import open_clip
    from PIL import Image

    print("  loading torch float source ...")
    if args.source is not None:
        model, _, _ = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained=args.source)
    else:
        model, _, _ = open_clip.create_model_and_transforms(
            "ViT-B-32", pretrained="laion2b_s34b_b79k")
    model.eval()
    tok = open_clip.get_tokenizer("ViT-B-32")

    print("  loading tflite candidate ...")
    # Interpreter order matters: TF first (full type support incl. int64
    # inputs + ARG_MAX). ai-edge-litert's Python build rejects int64
    # ("Unknown input type: 4") — an environment quirk, NOT a model flaw
    # (Android tflite_flutter runs int64 fine). Fall through on RUN failure,
    # not just on missing imports (2026-10-09).
    interp = None
    problems = []
    try:
        import tensorflow as tf
        interp = tf.lite.Interpreter(model_path=args.tflite)
        interp.allocate_tensors()
        print("  interpreter: tensorflow")
    except Exception as e:  # noqa: BLE001 — try the next runtime
        problems.append(f"tensorflow: {str(e)[:160]}")
        interp = None
    if interp is None:
        try:
            from tflite_runtime.interpreter import Interpreter as TRInterp
            interp = TRInterp(model_path=args.tflite)
            interp.allocate_tensors()
            print("  interpreter: tflite_runtime")
        except Exception as e:  # noqa: BLE001
            problems.append(f"tflite_runtime: {str(e)[:160]}")
            interp = None
    if interp is None:
        try:
            from ai_edge_litert.interpreter import Interpreter as EdgeInterp
            interp = EdgeInterp(model_path=args.tflite)
            interp.allocate_tensors()
            print("  interpreter: ai_edge_litert")
        except Exception as e:  # noqa: BLE001
            problems.append(f"ai_edge_litert: {str(e)[:160]}")
            interp = None
    if interp is None:
        print("FAIL: no interpreter could run the file:")
        for p in problems:
            print(f"    {p}")
        return 1
    in_det = interp.get_input_details()
    out_det = interp.get_output_details()
    print(f"  tflite in: {[(d['shape'].tolist(), str(d['dtype'])) for d in in_det]}")
    print(f"  tflite out: {[(d['shape'].tolist(), str(d['dtype'])) for d in out_det]}")
    # Find inputs by shape. NOTE: onnx2tf transposes the image tower to
    # NHWC ([1,224,224,3]) — TFLite convention — while the Dart frontend
    # emits NCHW, so both the validator and the loader transpose to NHWC.
    # (Int64-free graph: TFLite rejects int64 inputs and ArgMax outputs.)
    img_idx = txt_idx = eot_idx = None
    img_nhwc = False
    for d in in_det:
        s = list(d["shape"])
        if len(s) == 4 and (s[1] == 3 or s[3] == 3):
            img_idx = d["index"]
            img_nhwc = s[3] == 3
        elif len(s) == 2 and s[1] == 77:
            txt_idx = d["index"]
        elif (len(s) == 1 and s[0] == 1) or len(s) == 0:
            eot_idx = d["index"]
    outs512 = [d["index"] for d in out_det
               if len(list(d["shape"])) == 2 and list(d["shape"])[1] == 512]
    assert img_idx is not None and txt_idx is not None, "input layout moved!"
    assert eot_idx is not None, "eot input missing!"
    assert len(outs512) == 2, f"expected two 512-d heads, got {len(outs512)}"

    def tflite_embed(image_np, ids_np, eot):
        img = image_np.transpose(0, 2, 3, 1) if img_nhwc else image_np
        interp.set_tensor(img_idx, img.astype(np.float32))
        txt_dtype = [d for d in in_det if d["index"] == txt_idx][0]["dtype"]
        interp.set_tensor(txt_idx, ids_np.astype(
            np.int64 if "int64" in str(txt_dtype) else np.int32))
        interp.set_tensor(eot_idx, np.array([eot], dtype=np.int32))
        interp.invoke()
        return (interp.get_tensor(outs512[0])[0],
                interp.get_tensor(outs512[1])[0])

    print("  fetching probe photos ...")
    paths = fetch_images(args.cache)
    images = [clip_preprocess(Image.open(p)) for p in paths]
    with torch.no_grad():
        torch_txt = [
            model.encode_text(tok([c]), normalize=False)[0].numpy()
            for c in CAPTIONS
        ]
        torch_img = [
            model.encode_image(
                torch.from_numpy(im), normalize=False)[0].numpy()
            for im in images
        ]
    lite_txt, lite_img = [], []
    for c, im in zip(CAPTIONS, images):
        ids = tok([c]).numpy()
        row = ids[0].tolist()
        eot = row.index(49407, 1)  # first EOT at index>=1 (never in words)
        a, b = tflite_embed(im, ids, eot)
        lite_img.append(a)
        lite_txt.append(b)

    # Per-tower drift (nan-robust: report NaN fraction separately —
    # a NaN mean hides whether one tower or everything broke).
    drifts = []
    nan_towers = 0
    total_towers = 0
    for a, b in list(zip(torch_txt, lite_txt)) + list(zip(torch_img,
                                                          lite_img)):
        total_towers += 1
        na, nb = norm(np.asarray(a, dtype=np.float64)), norm(
            np.asarray(b, dtype=np.float64))
        if not np.all(np.isfinite(nb)):
            nan_towers += 1
            continue
        drifts.append(float(np.abs(na - nb).mean()))
    mean_drift = sum(drifts) / len(drifts) if drifts else float("nan")

    # Cross-matrix top-1 agreement (image -> caption).
    agree = 0
    for i in range(8):
        qt = [float(norm(torch_img[i]) @ norm(t)) for t in torch_txt]
        ql = [float(norm(np.asarray(lite_img[i])) @ norm(t))
              for t in lite_txt]
        if int(np.argmax(qt)) == int(np.argmax(ql)):
            agree += 1
    print(f"  mean abs drift: {mean_drift:.4f} (gate < 0.05)")
    print(f"  non-finite towers: {nan_towers}/{total_towers}")
    print(f"  top-1 agreement: {agree}/8 (gate >= 7)")
    if nan_towers == 0 and agree >= 7 and mean_drift < 0.05:
        print("PASS: int8 ranks like float — safe to ship to the phone")
        return 0
    print("FAIL: conversion damaged the space — do not ship")
    return 1


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
from huggingface_hub import hf_hub_download
src = hf_hub_download('laion/CLIP-ViT-B-32-laion2B-s34B-b79K',
                      'open_clip_pytorch_model.bin')
print('source:', src)


In [ ]:
!python export_joint_onnx.py --source "{src}" --out /content/clip_joint_b32.onnx

In [ ]:
# Float32 base (faithful lowering; --no-quantize skips onnx2tf's broken DR pass).
!python convert_onnx_to_tflite.py --onnx /content/clip_joint_b32.onnx --out /content/clip_joint_b32_f32.tflite --no-quantize
!ls -la /content/*.tflite

In [ ]:
# Reference dynamic-range (symmetric tables). Shippable candidate.
!python quantize_aeq.py --in /content/clip_joint_b32_float32.tflite --out /content/clip_joint_b32_aeq.tflite
!ls -la /content/*.tflite

In [ ]:
!python validate_clip.py --tflite /content/clip_joint_b32_aeq.tflite --source "{src}"

## Back on the PC (only if the gate prints PASS)

1. Download **`clip_joint_b32_aeq.tflite`** from Colab's Files pane.
2. Push it to the phone (renamed to the name the app expects):

```
adb push clip_joint_b32_aeq.tflite /sdcard/Android/data/com.example.querypic/files/models/clip_joint_b32_int8.tflite
```
(If push complains about ownership, push to `/data/local/tmp/` then copy it into the app's `app_flutter/models/` dir.)

3. In QueryPic: Settings > turn CLIP **off and on once** (resets the
   crash fuse), fully close the app, reopen.
4. Settings > Model status — paste the `clip:` lines back.
   Expect `joint loaded`, then a full re-index with real vectors.
